In [40]:
import matplotlib.pyplot as plt
import numpy as np
import os
import imageio

from parameter_file_interface import read_parameter_file

In [42]:
parfile = "photon_data/FMDiskRaytrace.par"
output_dir = "photon_data/data"
plot_output_dir = "photon_data/geodesic_rotation"
plot_output_frames_dir = "photon_data/geodesic_rotation/frames"

In [43]:
par = read_parameter_file(parfile, "RaytracingX")

num_pix_width = par["num_pixels_width"]
num_pix_height = par["num_pixels_height"]
camera_pos = par["camera_pos"]
x_min = -30
x_max = 30

In [44]:
num_photons = num_pix_width * num_pix_height

particle_pos = [[] for _ in range(num_photons)]
particle_tau = [[] for _ in range(num_photons)]
particle_lnE = [[] for _ in range(num_photons)]

for os_file in os.listdir(os.fsencode(output_dir)):
    filename = os.fsdecode(os_file)

    iter = int(filename[-5:])

    with open(output_dir + "/" + filename) as file:
        for i, line in enumerate(file):
            if i < 5:
                continue

            linesplit = line.split()

            #pos_x pos_y pox_z vel_lower_x vel_lower_y vel_lower_z ln_alphaE tau
            for j in [0, 1, 2, 5, 6, 7, 8, 9]:
                linesplit[j] = float(linesplit[j])
            #id cpu pixel_number
            for j in [3, 4, 10]:
                linesplit[j] = int(linesplit[j])

            particle_pos[linesplit[10]].append([linesplit[0], linesplit[1], linesplit[2]])
            particle_tau[linesplit[10]].append(linesplit[9])

max_iterations = np.max([len(particle_pos[i]) for i in range(num_photons)])
for i in range(num_photons):
    while len(particle_pos[i]) < max_iterations:
        particle_pos[i].append([np.nan, np.nan, np.nan])
        particle_tau[i].append(np.nan)

In [ ]:
fig = plt.figure(figsize=(9,9))

def makeplot(azimuth, data, num_lines):
    ax = fig.add_subplot(projection='3d')
    for i in range(num_lines):
        if i % 10 != 0:
            continue
        ax.plot(*np.array(data[i]).T, color='tab:blue')
    ax.view_init(azim=azimuth)

    ax.set_xlim([x_min, x_max])
    ax.set_ylim([x_min, x_max])
    ax.set_zlim([x_min, x_max])

    return ax

num_frames = 200
for i in range(num_frames):
    angle = i / num_frames * 360
    makeplot(angle, particle_pos, num_photons)
    plt.savefig(plot_output_frames_dir + "/frame%04d.png"%(i))
    fig.clear()

In [ ]:
images = []
filenames = []

for os_file in os.listdir(os.fsencode(plot_output_frames_dir)):
    if not os.fsdecode(os_file).endswith(".png"):
        continue

    filenames.append(os.fsdecode(os_file))

filenames.sort()

for filename in filenames:
    images.append(imageio.imread(plot_output_frames_dir + "/" + filename))
imageio.mimsave(plot_output_dir + "/geodesic_rotation.gif", images)

/var/folders/h4/l1j9m8_d36z3m4bmc5rbbv1r0000gn/T/ipykernel_99486/1392987274.py:13: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning dissapear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(plot_output_frames_dir + "/" + filename))
